# 司机周度得分 → 司机生命周期分析报告

本 Notebook 分步骤实现：
1. 读取 `司机分数趋势底表`
2. 计算站点级 & 大区级汇总指标 → sheet「站点汇总」
3. 生成「待召回优质司机」明细
4. 生成「新司机」明细
5. 导出到 `W33_Driver_Lifecycle_Report.xlsx`

## 关键假设（请先确认，不符合可在下方对应 cell 里改）

底表列是 `w-1 ~ w-4` 四周分数，**假设 w-1 = 最近一周（本周），w-2 = 上周，w-3/w-4 更早**。据此：

| 指标 | 定义 |
|---|---|
| 本周司机人数 | w-1 分数非空 |
| 上周司机人数 | w-2 分数非空 |
| 流失司机（总体） | w-2 非空 & w-1 空（上周有分数，本周无分数 ）|
| 总体司机流失率 | 流失司机数 / 上周司机人数 |
| 本周优质司机 | w-1 >= 90 |
| 优质司机流失 | w-2 >= 90 & w-1 空 |
| 优质司机流失率 | 优质司机流失人数 / 上周优质司机人数 |
| 新司机 | w-1 非空，且 w-2 & w-3 & w-4 全空（历史上首次出现） |
| 新司机占比 | 新司机人数 / 本周司机人数 |
| 本周已召回优质司机 | w-1 非空，w-2 空，且w-3 >= 90（曾为优质司机，中断后回归） |
| 上周待召回优质司机 | w-2 空，且w-3 >= 90 |
| 召回率 | 召回司机人数 / 待召回优质司机人数 |
| 本周待召回优质司机 | w-2 >=90 且 w-1 为空（曾经是优质司机、但本周流失）|

In [1]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


## 1. 配置路径
先在本地跑通，确认无误后再改成正式输出路径。

In [4]:
# ==== 按需修改 ====
INPUT_PATH = r"E:\GOFO Driver Dashboard\History\W39司机周度得分.xlsx"          # 输入文件路径（先放在 notebook 同目录方便测试）
SHEET_NAME = "司机分数趋势底表"

OUTPUT_DIR = r"E:\GOFO Driver Dashboard\History"
OUTPUT_NAME = "W39_Driver_Lifecycle_Report.xlsx"
GOOD_THRESHOLD = 90


## 2. 读取数据

In [5]:
df = pd.read_excel(INPUT_PATH, sheet_name=SHEET_NAME)
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()


(33171, 8)


,大区,站点,dsp,司机id,w-1司机分数,w-2司机分数,w-3司机分数,w-4司机分数
0,FL,APK01,APK-BML,14397,NaN,NaN,NaN,30.0
1,FL,APK01,APK-BML,15884,95.0,92.86,90.0,95.0
2,FL,APK01,APK-BML,23983,100.0,100.00,100.0,100.0
3,FL,APK01,APK-BML,205121,97.5,100.00,85.0,97.5
4,FL,APK01,APK-BML,206792,85.0,65.00,80.0,NaN


In [6]:
# 确认列名，如实际文件列名有差异（比如全角/半角、空格），在这里改映射
expected_cols = ["大区", "站点", "dsp", "司机id", "w-1司机分数", "w-2司机分数", "w-3司机分数", "w-4司机分数"]
missing = [c for c in expected_cols if c not in df.columns]
assert not missing, f"缺少列: {missing}, 实际列名: {list(df.columns)}"

for c in ["w-1司机分数", "w-2司机分数", "w-3司机分数", "w-4司机分数"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


## 3. 计算司机粒度的辅助布尔列
所有汇总指标都基于这些逐司机的 True/False 列做 sum，逻辑集中在这一步，方便核对。

In [7]:
w1, w2, w3, w4 = "w-1司机分数", "w-2司机分数", "w-3司机分数", "w-4司机分数"

df["has_w1"] = df[w1].notna()
df["has_w2"] = df[w2].notna()
df["has_w3"] = df[w3].notna()
df["has_w4"] = df[w4].notna()

df["is_good_w1"] = df["has_w1"] & (df[w1] >= GOOD_THRESHOLD)
df["is_good_w2"] = df["has_w2"] & (df[w2] >= GOOD_THRESHOLD)
df["is_good_w3"] = df["has_w3"] & (df[w3] >= GOOD_THRESHOLD)
df["is_good_w4"] = df["has_w4"] & (df[w4] >= GOOD_THRESHOLD)

# 流失：上周在岗，本周不在岗
df["churned"] = df["has_w2"] & ~df["has_w1"]

# 优质流失：上周是优质（w-2>=90），本周没有分数（流失）
df["good_churned"] = df["is_good_w2"] & ~df["has_w1"]

# 新司机：本周在岗，且过去三周从未出现过
df["new_driver"] = df["has_w1"] & ~df["has_w2"] & ~df["has_w3"] & ~df["has_w4"]

# 已召回优质：本周在岗，上周不在岗，但更早（w-3或w-4）是优质司机
df["recalled"] = df["has_w1"] & ~df["has_w2"] & df["is_good_w3"]

# 上周待召回优质司机：w-3或w-4是优质，w-2没有分数（这是召回率的分母）
df["pending_recall_last_week"] = df["is_good_w3"] & ~df["has_w2"]

df[["大区","站点","司机id",w1,w2,w3,w4,"churned","good_churned","new_driver","recalled","pending_recall_last_week"]].head(10)


,大区,站点,司机id,w-1司机分数,w-2司机分数,w-3司机分数,w-4司机分数,churned,good_churned,new_driver,recalled,pending_recall_last_week
0,FL,APK01,14397,NaN,NaN,NaN,30.0,False,False,False,False,False
1,FL,APK01,15884,95.0,92.86,90.00,95.0,False,False,False,False,False
2,FL,APK01,23983,100.0,100.00,100.00,100.0,False,False,False,False,False
3,FL,APK01,205121,97.5,100.00,85.00,97.5,False,False,False,False,False
4,FL,APK01,206792,85.0,65.00,80.00,NaN,False,False,False,False,False
5,FL,APK01,248787,NaN,NaN,NaN,100.0,False,False,False,False,False
6,FL,APK01,271468,100.0,80.00,66.67,97.5,False,False,False,False,False
7,FL,APK01,391387,NaN,NaN,85.00,NaN,False,False,False,False,False
8,FL,APK01,398289,NaN,NaN,NaN,100.0,False,False,False,False,False
9,FL,APK01,399381,90.0,90.00,87.50,90.0,False,False,False,False,False


## 4. 定义汇总函数（可复用于站点级和大区级）
用 `groupby().agg()` 做布尔列求和，再用向量化运算算比率，兼容各版本 pandas，速度也更快。

In [8]:
def safe_div(numer: pd.Series, denom: pd.Series) -> pd.Series:
    # 分母为0时返回 NaN（而不是 inf），避免除0报错/脏数据
    return (numer / denom).replace([np.inf, -np.inf], np.nan)

def summarize_by(df_in: pd.DataFrame, group_cols: list) -> pd.DataFrame:
    agg = df_in.groupby(group_cols, as_index=False).agg(
        本周司机人数=("has_w1", "sum"),
        上周司机人数=("has_w2", "sum"),
        总体流失司机数=("churned", "sum"),
        本周优质司机人数=("is_good_w1", "sum"),
        上周优质司机人数=("is_good_w2", "sum"),
        优质司机流失人数=("good_churned", "sum"),
        新司机人数=("new_driver", "sum"),
        上周待召回优质司机人数=("pending_recall_last_week","sum"),
        已召回优质司机人数=("recalled", "sum")
    )

    agg["总体司机流失率"] = safe_div(agg["总体流失司机数"], agg["上周司机人数"])
    agg["优质司机流失率"] = safe_div(agg["优质司机流失人数"], agg["上周优质司机人数"])
    agg["新司机占比"] = safe_div(agg["新司机人数"], agg["本周司机人数"])
    # 召回率分母 = 上周待召回优质司机人数（w-3优质、w-2无分数）
    agg["召回率"] = safe_div(agg["已召回优质司机人数"], agg["上周待召回优质司机人数"])

    agg = agg.drop(columns=["上周优质司机人数"])  # 只是中间计算用，最终报表不需要展示
    return agg


## 5. 站点级汇总 + 每个大区末尾插入大区汇总行

In [9]:
station_summary = summarize_by(df, ["大区", "站点"])
station_summary.head()


,大区,站点,本周司机人数,上周司机人数,总体流失司机数,本周优质司机人数,优质司机流失人数,新司机人数,上周待召回优质司机人数,已召回优质司机人数,总体司机流失率,优质司机流失率,新司机占比,召回率
0,FL,APK01,241,257,53,113,6,32,5,1,0.206226,0.053571,0.132780,0.2
1,FL,FLL01,389,393,45,199,11,31,5,0,0.114504,0.050691,0.079692,0.0
2,FL,FPR01,96,103,25,49,5,18,2,0,0.242718,0.096154,0.187500,0.0
3,FL,FTM01,175,172,19,104,2,16,2,0,0.110465,0.020408,0.091429,0.0
4,FL,JAX01,189,201,32,95,5,15,2,0,0.159204,0.049505,0.079365,0.0


In [10]:
region_summary = summarize_by(df, ["大区"])
region_summary["站点"] = "大区汇总"

# 列顺序对齐，方便后面 concat
col_order = ["大区", "站点", "本周司机人数", "上周司机人数", "总体流失司机数", "总体司机流失率",
             "本周优质司机人数", "优质司机流失人数", "优质司机流失率",
             "新司机人数", "新司机占比", "上周待召回优质司机人数", "已召回优质司机人数", "召回率"]
station_summary = station_summary[col_order]
region_summary = region_summary[col_order]

# 按大区插入汇总行：每个大区的站点明细结束后，紧跟一行“大区汇总”
final_rows = []
for region in station_summary["大区"].unique():
    final_rows.append(station_summary[station_summary["大区"] == region])
    final_rows.append(region_summary[region_summary["大区"] == region])

site_report = pd.concat(final_rows, ignore_index=True)
site_report.head()


,大区,站点,本周司机人数,上周司机人数,总体流失司机数,总体司机流失率,本周优质司机人数,优质司机流失人数,优质司机流失率,新司机人数,新司机占比,上周待召回优质司机人数,已召回优质司机人数,召回率
0,FL,APK01,241,257,53,0.206226,113,6,0.053571,32,0.132780,5,1,0.2
1,FL,FLL01,389,393,45,0.114504,199,11,0.050691,31,0.079692,5,0,0.0
2,FL,FPR01,96,103,25,0.242718,49,5,0.096154,18,0.187500,2,0,0.0
3,FL,FTM01,175,172,19,0.110465,104,2,0.020408,16,0.091429,2,0,0.0
4,FL,JAX01,189,201,32,0.159204,95,5,0.049505,15,0.079365,2,0,0.0


## 6. 「待召回优质司机」明细：上周优质、本周流失

In [11]:
cols_out = ["大区", "站点", "dsp", "司机id", w1, w2, w3, w4]

pending_recall = df[df["is_good_w2"] & ~df["has_w1"]][cols_out].reset_index(drop=True)
pending_recall


,大区,站点,dsp,司机id,w-1司机分数,w-2司机分数,w-3司机分数,w-4司机分数
0,FL,APK01,APK-HDW,200402,NaN,90.0,90.0,100.0
1,FL,APK01,APK-HDW,268551,NaN,90.0,NaN,NaN
2,FL,APK01,APK-HDW,433227,NaN,100.0,95.0,NaN
3,FL,APK01,APK-JET,255073,NaN,90.0,100.0,80.0
4,FL,APK01,APK-JNJ,267075,NaN,95.0,NaN,75.0
...,...,...,...,...,...,...,...,...
568,WE,SMF01,SMF-DAE,376092,NaN,90.0,95.0,60.0
569,WE,SMF01,SMF-RRI,239659,NaN,100.0,85.0,95.0
570,WE,STG01,STG-CJJ,309835,NaN,90.0,100.0,87.5
571,WE,TRC01,TRC-YLU,17140,NaN,100.0,97.5,85.0


## 7. 「新司机」明细

In [12]:
new_drivers_detail = df[df["new_driver"]][cols_out].reset_index(drop=True)
new_drivers_detail


,大区,站点,dsp,司机id,w-1司机分数,w-2司机分数,w-3司机分数,w-4司机分数
0,FL,APK01,APK-DRT,403422,0.00,NaN,NaN,NaN
1,FL,APK01,APK-DRT,422444,71.00,NaN,NaN,NaN
2,FL,APK01,APK-DRT,442419,91.63,NaN,NaN,NaN
3,FL,APK01,APK-DRT,442781,93.33,NaN,NaN,NaN
4,FL,APK01,APK-EPC,403168,85.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2974,WE,VTC01,VTC-RWV,444698,95.00,NaN,NaN,NaN
2975,WE,VTC01,VTC-RWV,446216,60.00,NaN,NaN,NaN
2976,WE,VTC01,VTC-RWV,447565,70.00,NaN,NaN,NaN
2977,WE,VTC01,VTC-RWV,448584,60.00,NaN,NaN,NaN


## 8. 导出 Excel（含百分比格式 + 大区汇总行加粗）

In [13]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, OUTPUT_NAME)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    site_report.to_excel(writer, sheet_name="站点汇总", index=False)
    pending_recall.to_excel(writer, sheet_name="本周待召回优质司机", index=False)
    new_drivers_detail.to_excel(writer, sheet_name="新司机", index=False)
    
print("已保存:", output_path)


已保存: E:\GOFO Driver Dashboard\History\W39_Driver_Lifecycle_Report.xlsx


In [14]:
# 补充格式：百分比列、大区汇总行加粗、列宽自适应
wb = load_workbook(output_path)
ws = wb["站点汇总"]

pct_cols = ["总体司机流失率", "优质司机流失率", "新司机占比", "召回率"]
header = [cell.value for cell in ws[1]]
pct_col_idx = [header.index(c) + 1 for c in pct_cols if c in header]

bold_font = Font(bold=True)
subtotal_fill = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")

for row in range(2, ws.max_row + 1):
    for col_idx in pct_col_idx:
        cell = ws.cell(row=row, column=col_idx)
        if isinstance(cell.value, (int, float)):
            cell.number_format = "0.0%"
    site_cell_idx = header.index("站点") + 1
    if ws.cell(row=row, column=site_cell_idx).value == "大区汇总":
        for col_idx in range(1, ws.max_column + 1):
            c = ws.cell(row=row, column=col_idx)
            c.font = bold_font
            c.fill = subtotal_fill

# 简单列宽自适应（所有 sheet）
for sheet in wb.worksheets:
    for col_cells in sheet.columns:
        length = max((len(str(c.value)) if c.value is not None else 0) for c in col_cells)
        col_letter = get_column_letter(col_cells[0].column)
        sheet.column_dimensions[col_letter].width = min(max(length + 2, 10), 30)

wb.save(output_path)
print("格式化完成:", output_path)


格式化完成: E:\GOFO Driver Dashboard\History\W39_Driver_Lifecycle_Report.xlsx


## 9. 快速核对（可选）

跑完以上步骤后，建议抽查几行：
- 随手挑一个司机在原始底表里对一下 w-1~w-4，看他/她是否被正确归到"流失/新司机/召回/正常"。
- 检查每个大区汇总行的"本周司机人数"是否等于该大区下所有站点行之和。


In [15]:
# 抽查：大区汇总的本周司机人数 是否等于各站点之和
def check_region(region):
    stations_sum = site_report[(site_report["大区"] == region) & (site_report["站点"] != "大区汇总")]["本周司机人数"].sum()
    region_total = site_report[(site_report["大区"] == region) & (site_report["站点"] == "大区汇总")]["本周司机人数"].iloc[0]
    return stations_sum == region_total

pd.Series({r: check_region(r) for r in site_report["大区"].unique()})


FL    True
GL    True
MS    True
NE    True
TX    True
WE    True
dtype: bool